In [1]:
!pip install yt-dlp opencv-python transformers torchvision pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.5 MB/s eta 0:00:00


In [2]:
import os

url = "https://www.youtube.com/watch?v=BB49x_uMlGA"  # sample video
os.system(f"yt-dlp -f mp4 -o video.mp4 {url}")

0

In [3]:
import cv2
import torch
import torch.nn as nn
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load BLIP
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# Extract frames (uniform)
def extract_frames(video_path, interval=60):
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % interval == 0:
            frames.append(frame)
        count += 1
    cap.release()
    return frames

# Caption
def caption_frame(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=20)
    return processor.decode(out[0], skip_special_tokens=True)

frames = extract_frames("video.mp4")
captions = [caption_frame(f) for f in frames]

print("\nFrame Captions:")
for c in captions:
    print("-", c)

# ---- LSTM ----
class CaptionLSTM(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, 256, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return out[:, -1, :]

lstm_model = CaptionLSTM()

# simple tokenizer
word_to_idx = {}
def encode(text):
    tokens = text.split()
    idxs = []
    for t in tokens:
        if t not in word_to_idx:
            word_to_idx[t] = len(word_to_idx)+1
        idxs.append(word_to_idx[t])
    return torch.tensor(idxs)

encoded = [encode(c) for c in captions]
padded = nn.utils.rnn.pad_sequence(encoded, batch_first=True)

_ = lstm_model(padded)

print("\nTemporal modeling applied (LSTM)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]


Frame Captions:
- a man walking a dog
- a man walking a dog
- a man is playing with a dog in a field
- a man is walking in the grass
- a man walking a dog

Temporal modeling applied (LSTM)


In [ ]:
import cv2
import numpy as np
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# Keyframe extraction
def extract_keyframes(video_path, threshold=30):
    cap = cv2.VideoCapture(video_path)
    prev = None
    keyframes = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev is None:
            keyframes.append(frame)
            prev = gray
            continue

        diff = cv2.absdiff(prev, gray)
        score = np.mean(diff)

        if score > threshold:
            keyframes.append(frame)

        prev = gray

    cap.release()
    return keyframes

def caption_frame(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=20)
    return processor.decode(out[0], skip_special_tokens=True)

keyframes = extract_keyframes("video.mp4")
print("Keyframes:", len(keyframes))

captions = [caption_frame(f) for f in keyframes]

print("\nKeyframe Captions:")
for c in captions:
    print("-", c)

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from collections import Counter

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

# --- Keyframes ---
def extract_keyframes(video_path, threshold=30):
    cap = cv2.VideoCapture(video_path)
    prev = None
    keyframes = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev is None:
            keyframes.append(frame)
            prev = gray
            continue

        diff = cv2.absdiff(prev, gray)
        score = np.mean(diff)

        if score > threshold:
            keyframes.append(frame)

        prev = gray

    cap.release()
    return keyframes

# --- Caption ---
def caption_frame(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=20)
    return processor.decode(out[0], skip_special_tokens=True)

# --- LSTM ---
class CaptionLSTM(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, 256, batch_first=True)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return out[:, -1, :]

lstm_model = CaptionLSTM()

word_to_idx = {}
def encode(text):
    tokens = text.split()
    idxs = []
    for t in tokens:
        if t not in word_to_idx:
            word_to_idx[t] = len(word_to_idx)+1
        idxs.append(word_to_idx[t])
    return torch.tensor(idxs)

# --- PIPELINE ---
video_path = "video.mp4"

keyframes = extract_keyframes(video_path)
captions = [caption_frame(f) for f in keyframes]

print("\nCaptions:")
for c in captions:
    print("-", c)

# Temporal modeling
encoded = [encode(c) for c in captions]
padded = nn.utils.rnn.pad_sequence(encoded, batch_first=True)
_ = lstm_model(padded)

# Final summary
summary = Counter(captions).most_common(5)

print("\nFinal Video Description:")
for s, _ in summary:
    print("-", s)